<a href="https://colab.research.google.com/github/alexacoonline/7006SCN_CAC_17089427/blob/main/Task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark datasets pyarrow

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("FineWeb_BigData") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")


from datasets import load_dataset
import pyarrow as pa
import pyarrow.parquet as pq

dataset = load_dataset(
    "HuggingFaceFW/fineweb",
    split="train",
    streaming=True
)


CHUNK_SIZE = 200000   # write every 200k rows
MAX_ROWS = 1000000    # 1M sample from dataset
buffer = []
file_index = 0
count = 0

for row in dataset:
    buffer.append(row)
    count += 1

    if len(buffer) >= CHUNK_SIZE:
        table = pa.Table.from_pylist(buffer)
        pq.write_table(table, f"fineweb_chunk_{file_index}.parquet")
        buffer = []
        file_index += 1
        print(f"Saved chunk {file_index}")

    if count >= MAX_ROWS:
        break

# write remaining
if buffer:
    table = pa.Table.from_pylist(buffer)
    pq.write_table(table, f"fineweb_chunk_{file_index}.parquet")

print("Streaming + storage complete")


df = spark.read.parquet("fineweb_chunk_*.parquet")

df.printSchema()
df.show(5)


